In [2]:
import requests
import re
import json
import pandas as pd
import time
from urllib.parse import urlparse
from bs4 import BeautifulSoup
from dotenv import load_dotenv
import os

In [3]:
# 1. Obtener el HTML de la página del ranking
load_dotenv()
url = os.getenv("TARGET_MORTGAGE_INTEREST_RATE")
BASE_URL = os.getenv("BASE_URL")

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers)
response.raise_for_status()  # Lanza error si la petición falla

In [4]:
# 2. Parsear HTML con BeautifulSoup
soup = BeautifulSoup(response.text, "html.parser")

In [5]:
# 3. Buscar la tabla principal y extraer los links
table = soup.find("table")
country_links = []


if table:
    rows = table.find_all("tr")
    for row in rows:
        cell = row.find("td")
        if cell and cell.a:
            relative_link = cell.a["href"]
            full_url = f"{BASE_URL}{relative_link}"
            country_links.append(full_url)

# Convertir la lista en DataFrame
df_links = pd.DataFrame(country_links, columns=["url"])

# Guardar como archivo Parquet
df_links.to_parquet("../../data/country_mortgage_interest_links.parquet", index=False)

In [6]:
# 4. Mostrar resultado
print("Links encontrados:", df_links.size)

Links encontrados: 83


In [7]:
# configuracion necesaria para descargar datos

master_df = pd.DataFrame()

pattern = r'arrayToDataTable\(\s*(\[\[.*?\]\])\s*\);'

sleep_duration = 5

In [8]:
# Cargar los países válidos desde el archivo parquet
df_countries = pd.read_parquet("../../data/countries_with_hpi.parquet")
valid_countries = set(df_countries["country"].unique())

In [9]:
# DataFrames de salida
dfs_quarters = []
dfs_months = []

start_time = time.time()
for url in country_links:
    country = urlparse(url).path.strip("/").split("/")[0]

    if country not in valid_countries:
        print(f"⏭️ País {country} no está en la lista, se omite.")
        continue

    try:
        response = requests.get(url, headers=headers)
        html = response.text

        match = re.search(pattern, html, re.DOTALL)
        if match:
            data_str = match.group(1)
            data = json.loads(data_str)

            df = pd.DataFrame(data[1:], columns=data[0])
            df.columns = ["Period", "Value"]
            df["Country"] = country

            # --- separar según formato de Period ---
            if df["Period"].str.contains(r"Q\d").any():
                df_pivot = df.pivot(index="Period", columns="Country", values="Value")
                dfs_quarters.append(df_pivot)
                print(f"✅ Datos trimestrales extraídos para {country}")
            else:
                df_pivot = df.pivot(index="Period", columns="Country", values="Value")
                dfs_months.append(df_pivot)
                print(f"✅ Datos mensuales extraídos para {country}")
        else:
            print(f"⚠️ No se encontraron datos en {url}")
    except Exception as e:
        print(f"❌ Error procesando {url}: {e}")
    
    time.sleep(sleep_duration)

# unir cada grupo por separado
df_quarters = pd.concat(dfs_quarters, axis=1) if dfs_quarters else pd.DataFrame()
df_months = pd.concat(dfs_months, axis=1) if dfs_months else pd.DataFrame()

end_time = time.time()
total_sleep_time = len(valid_countries) * sleep_duration
effective_time = end_time - start_time - total_sleep_time

print(f"⏳ Tiempo total (incluyendo esperas): {end_time - start_time:.2f} s")
print(f"😴 Tiempo en espera: {total_sleep_time:.2f} s")
print(f"⚡ Tiempo efectivo de ejecución: {effective_time:.2f} s")

⏭️ País Albania no está en la lista, se omite.
⏭️ País Antigua-and-Barbuda no está en la lista, se omite.
⏭️ País Argentina no está en la lista, se omite.
⏭️ País Armenia no está en la lista, se omite.
⏭️ País Aruba no está en la lista, se omite.
✅ Datos mensuales extraídos para Australia
✅ Datos mensuales extraídos para Austria
⏭️ País Bahamas no está en la lista, se omite.
⏭️ País Bahrain no está en la lista, se omite.
⏭️ País Bangladesh no está en la lista, se omite.
✅ Datos mensuales extraídos para Belgium
⏭️ País Benin no está en la lista, se omite.
⏭️ País Bhutan no está en la lista, se omite.
⏭️ País Bosnia-and-Herzegovina no está en la lista, se omite.
✅ Datos mensuales extraídos para Brazil
✅ Datos mensuales extraídos para Bulgaria
⏭️ País Burkina-Faso no está en la lista, se omite.
✅ Datos mensuales extraídos para Canada
✅ Datos mensuales extraídos para Chile
⏭️ País Costa-Rica no está en la lista, se omite.
✅ Datos mensuales extraídos para Croatia
✅ Datos mensuales extraídos

In [10]:
# resultados finales
print("\n📊 DataFrame trimestral")
df_quarters.head()


📊 DataFrame trimestral


""


In [11]:
print("\n📊 DataFrame mensual")
df_months.head()


📊 DataFrame mensual


Country,Australia,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,Croatia,Cyprus,Czech-Republic,...,Slovakia,Slovenia,South-Africa,South-Korea,Spain,Sweden,Switzerland,Turkey,USA,United-Kingdom
Period,,,,,,,,,,,,,,,,,,,,,
Apr 2000,7.70,5.73,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,14.5,NaN,NaN,NaN,NaN,NaN,7.96,NaN
Apr 2001,6.58,6.12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,14.5,NaN,NaN,NaN,NaN,NaN,6.88,NaN
Apr 2002,6.45,5.36,NaN,NaN,NaN,NaN,7.18,NaN,NaN,7.61,...,NaN,NaN,15.0,6.69,NaN,NaN,NaN,49.46,6.62,NaN
Apr 2003,6.33,4.50,5.31,NaN,NaN,NaN,4.74,NaN,NaN,6.90,...,NaN,NaN,17.0,6.49,4.54,NaN,NaN,50.24,5.46,NaN
Apr 2004,7.00,3.85,4.74,NaN,NaN,NaN,3.99,NaN,NaN,6.26,...,NaN,NaN,11.5,6.03,3.61,NaN,NaN,23.87,5.68,4.71


---

In [12]:
# obtain paraguay url
template_url = country_links[0]
country = "Paraguay"

# Separar por "/"
parts = template_url.strip("/").split("/")

# Reemplazar el país (posición 3: después del dominio)

parts[3] = country

# Reconstruir la URL
custom_url = "/".join(parts) + "/"

In [13]:
# URL específica para Paraguay
response = requests.get(custom_url, headers=headers)
html = response.text

# Buscar el bloque JSON dentro del HTML
match = re.search(pattern, html, re.DOTALL)
if match:
    data_str = match.group(1)
    data = json.loads(data_str)

    # Crear DataFrame con los datos
    df = pd.DataFrame(data[1:], columns=data[0])
    df.columns = ["Period", "Value"]
    df["Country"] = "Paraguay"  # Asignar explícitamente

    # Pivotear para formato ancho (opcional)
    df_paraguay = df.pivot(index="Period", columns="Country", values="Value")

    print("✅ Datos de Paraguay extraídos correctamente")
else:
    print("⚠️ No se encontraron datos en el HTML para Paraguay")

✅ Datos de Paraguay extraídos correctamente


In [14]:
df_paraguay.head()

Country,Paraguay
Period,
Apr 2003,39.88
Apr 2004,35.68
Apr 2005,26.30
Apr 2006,19.88
Apr 2007,15.91


In [15]:
# Unir ambos DataFrames por el índice 'Period', preservando todos los datos
new_month_df = pd.merge(df_months, df_paraguay, how="outer", left_index=True, right_index=True)

In [17]:
new_month_df.tail()

Country,Australia,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,Croatia,Cyprus,Czech-Republic,...,Slovenia,South-Africa,South-Korea,Spain,Sweden,Switzerland,Turkey,USA,United-Kingdom,Paraguay
Period,,,,,,,,,,,,,,,,,,,,,
Sep 2020,3.77,1.30,1.83,7.14,3.27,2.77,2.35,3.48,1.68,2.51,...,2.40,7.00,2.44,1.21,1.55,1.28,14.78,2.65,1.71,10.53
Sep 2021,3.65,1.21,1.76,7.20,3.06,2.40,3.29,3.16,1.74,2.40,...,2.18,7.00,3.01,1.10,1.42,1.38,17.91,2.65,1.85,10.11
Sep 2022,6.36,2.11,1.90,10.25,2.84,3.58,4.32,2.94,2.22,2.62,...,2.20,9.75,4.79,1.54,2.09,3.10,20.42,6.33,2.96,9.90
Sep 2023,7.57,3.85,2.97,10.20,2.81,4.63,4.35,2.88,3.66,3.07,...,3.83,11.75,4.35,3.50,3.80,2.94,42.21,7.02,5.39,9.85
Sep 2024,7.58,3.88,3.40,9.46,2.79,4.73,4.81,3.07,3.99,3.48,...,4.37,11.50,3.74,3.55,3.93,1.97,42.93,5.62,5.21,9.71


In [18]:
unique_periods = pd.Series(new_month_df.index.astype(str)).unique()
print(unique_periods)

['Apr 1965' 'Apr 1966' 'Apr 1967' 'Apr 1968' 'Apr 1969' 'Apr 1970'
 'Apr 1971' 'Apr 1972' 'Apr 1973' 'Apr 1974' 'Apr 1975' 'Apr 1976'
 'Apr 1977' 'Apr 1978' 'Apr 1979' 'Apr 1980' 'Apr 1981' 'Apr 1982'
 'Apr 1983' 'Apr 1984' 'Apr 1985' 'Apr 1986' 'Apr 1987' 'Apr 1988'
 'Apr 1989' 'Apr 1990' 'Apr 1991' 'Apr 1992' 'Apr 1993' 'Apr 1994'
 'Apr 1995' 'Apr 1996' 'Apr 1997' 'Apr 1998' 'Apr 1999' 'Apr 2000'
 'Apr 2001' 'Apr 2002' 'Apr 2003' 'Apr 2004' 'Apr 2005' 'Apr 2006'
 'Apr 2007' 'Apr 2008' 'Apr 2009' 'Apr 2010' 'Apr 2011' 'Apr 2012'
 'Apr 2013' 'Apr 2014' 'Apr 2015' 'Apr 2016' 'Apr 2017' 'Apr 2018'
 'Apr 2019' 'Apr 2020' 'Apr 2021' 'Apr 2022' 'Apr 2023' 'Apr 2024'
 'Apr 2025' 'Aug 1965' 'Aug 1966' 'Aug 1967' 'Aug 1968' 'Aug 1969'
 'Aug 1970' 'Aug 1971' 'Aug 1972' 'Aug 1973' 'Aug 1974' 'Aug 1975'
 'Aug 1976' 'Aug 1977' 'Aug 1978' 'Aug 1979' 'Aug 1980' 'Aug 1981'
 'Aug 1982' 'Aug 1983' 'Aug 1984' 'Aug 1985' 'Aug 1986' 'Aug 1987'
 'Aug 1988' 'Aug 1989' 'Aug 1990' 'Aug 1991' 'Aug 1992' 'Aug 1

In [19]:
def split_period_index(df):
    """
    Recibe un DataFrame con índice 'Period' (ej: 'Apr 1960' o 'Q1 1990')
    Devuelve el mismo DataFrame con columnas 'Year' y 'Month' o 'Quarter'
    """
    df = df.copy().reset_index()  # sacar Period del índice
    df["Year"] = None
    df["Month"] = None
    df["Quarter"] = None

    for i, period in enumerate(df["Period"]):
        parts = period.split()

        if len(parts) == 2:
            first, year = parts
            df.at[i, "Year"] = int(year)

            if "Q" in first:  # Formato trimestral
                df.at[i, "Quarter"] = first
            else:  # Formato mensual
                df.at[i, "Month"] = first

    return df.set_index("Period")  # opcional: devolver con Period otra vez de índice

In [20]:
new_month_df = split_period_index(new_month_df)

In [21]:
new_month_df.head()

Country,Australia,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,Croatia,Cyprus,Czech-Republic,...,Spain,Sweden,Switzerland,Turkey,USA,United-Kingdom,Paraguay,Year,Month,Quarter
Period,,,,,,,,,,,,,,,,,,,,,
Apr 1965,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1965,Apr,None
Apr 1966,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1966,Apr,None
Apr 1967,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1967,Apr,None
Apr 1968,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1968,Apr,None
Apr 1969,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1969,Apr,None


In [22]:
new_month_df.tail()

Country,Australia,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,Croatia,Cyprus,Czech-Republic,...,Spain,Sweden,Switzerland,Turkey,USA,United-Kingdom,Paraguay,Year,Month,Quarter
Period,,,,,,,,,,,,,,,,,,,,,
Sep 2020,3.77,1.30,1.83,7.14,3.27,2.77,2.35,3.48,1.68,2.51,...,1.21,1.55,1.28,14.78,2.65,1.71,10.53,2020,Sep,None
Sep 2021,3.65,1.21,1.76,7.20,3.06,2.40,3.29,3.16,1.74,2.40,...,1.10,1.42,1.38,17.91,2.65,1.85,10.11,2021,Sep,None
Sep 2022,6.36,2.11,1.90,10.25,2.84,3.58,4.32,2.94,2.22,2.62,...,1.54,2.09,3.10,20.42,6.33,2.96,9.90,2022,Sep,None
Sep 2023,7.57,3.85,2.97,10.20,2.81,4.63,4.35,2.88,3.66,3.07,...,3.50,3.80,2.94,42.21,7.02,5.39,9.85,2023,Sep,None
Sep 2024,7.58,3.88,3.40,9.46,2.79,4.73,4.81,3.07,3.99,3.48,...,3.55,3.93,1.97,42.93,5.62,5.21,9.71,2024,Sep,None


In [23]:
def month_to_quarter(df):
    """
    Takes a DataFrame with 'Month' and 'Quarter' columns.
    If 'Month' exists, it assigns the corresponding quarter to 'Quarter'.
    """
    df = df.copy()

    # Dictionary for mapping months → quarters
    month_to_q = {
        "Jan": 1, "Feb": 1, "Mar": 1,
        "Apr": 2, "May": 2, "June": 2,
        "Jul": 3, "Aug": 3, "Sep": 3,
        "Oct": 4, "Nov": 4, "Dec": 4,
    }

    # Update quarter only where Month exists
    df["Quarter"] = df.apply(
        lambda row: month_to_q.get(row["Month"], row["Quarter"]),
        axis=1
    )

    return df

In [24]:
new_month_df = month_to_quarter(new_month_df)

In [25]:
new_month_df.tail()

Country,Australia,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,Croatia,Cyprus,Czech-Republic,...,Spain,Sweden,Switzerland,Turkey,USA,United-Kingdom,Paraguay,Year,Month,Quarter
Period,,,,,,,,,,,,,,,,,,,,,
Sep 2020,3.77,1.30,1.83,7.14,3.27,2.77,2.35,3.48,1.68,2.51,...,1.21,1.55,1.28,14.78,2.65,1.71,10.53,2020,Sep,3
Sep 2021,3.65,1.21,1.76,7.20,3.06,2.40,3.29,3.16,1.74,2.40,...,1.10,1.42,1.38,17.91,2.65,1.85,10.11,2021,Sep,3
Sep 2022,6.36,2.11,1.90,10.25,2.84,3.58,4.32,2.94,2.22,2.62,...,1.54,2.09,3.10,20.42,6.33,2.96,9.90,2022,Sep,3
Sep 2023,7.57,3.85,2.97,10.20,2.81,4.63,4.35,2.88,3.66,3.07,...,3.50,3.80,2.94,42.21,7.02,5.39,9.85,2023,Sep,3
Sep 2024,7.58,3.88,3.40,9.46,2.79,4.73,4.81,3.07,3.99,3.48,...,3.55,3.93,1.97,42.93,5.62,5.21,9.71,2024,Sep,3


In [27]:
new_month_df.dtypes

Country
Australia         float64
Austria           float64
Belgium           float64
Brazil            float64
Bulgaria          float64
Canada            float64
Chile             float64
Croatia           float64
Cyprus            float64
Czech-Republic    float64
Denmark           float64
Estonia           float64
Finland           float64
France            float64
Germany           float64
Greece            float64
Hungary           float64
Ireland           float64
Italy             float64
Latvia            float64
Lithuania         float64
Luxembourg        float64
Macedonia         float64
Malta             float64
Netherlands       float64
New-Zealand       float64
Norway            float64
Peru              float64
Poland            float64
Portugal          float64
Romania           float64
Russia            float64
Serbia            float64
Slovakia          float64
Slovenia          float64
South-Africa      float64
South-Korea       float64
Spain             float64
Swed

In [28]:
def year_to_int(df):
    """
    Convierte la columna 'Year' a enteros.
    Si hay valores no numéricos, los convierte en NaN.
    """
    df = df.copy()
    df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype("Int64")
    return df

In [29]:
new_month_df = year_to_int(new_month_df)
new_month_df.dtypes

Country
Australia         float64
Austria           float64
Belgium           float64
Brazil            float64
Bulgaria          float64
Canada            float64
Chile             float64
Croatia           float64
Cyprus            float64
Czech-Republic    float64
Denmark           float64
Estonia           float64
Finland           float64
France            float64
Germany           float64
Greece            float64
Hungary           float64
Ireland           float64
Italy             float64
Latvia            float64
Lithuania         float64
Luxembourg        float64
Macedonia         float64
Malta             float64
Netherlands       float64
New-Zealand       float64
Norway            float64
Peru              float64
Poland            float64
Portugal          float64
Romania           float64
Russia            float64
Serbia            float64
Slovakia          float64
Slovenia          float64
South-Africa      float64
South-Korea       float64
Spain             float64
Swed

In [30]:
new_month_df.tail()

Country,Australia,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,Croatia,Cyprus,Czech-Republic,...,Spain,Sweden,Switzerland,Turkey,USA,United-Kingdom,Paraguay,Year,Month,Quarter
Period,,,,,,,,,,,,,,,,,,,,,
Sep 2020,3.77,1.30,1.83,7.14,3.27,2.77,2.35,3.48,1.68,2.51,...,1.21,1.55,1.28,14.78,2.65,1.71,10.53,2020,Sep,3
Sep 2021,3.65,1.21,1.76,7.20,3.06,2.40,3.29,3.16,1.74,2.40,...,1.10,1.42,1.38,17.91,2.65,1.85,10.11,2021,Sep,3
Sep 2022,6.36,2.11,1.90,10.25,2.84,3.58,4.32,2.94,2.22,2.62,...,1.54,2.09,3.10,20.42,6.33,2.96,9.90,2022,Sep,3
Sep 2023,7.57,3.85,2.97,10.20,2.81,4.63,4.35,2.88,3.66,3.07,...,3.50,3.80,2.94,42.21,7.02,5.39,9.85,2023,Sep,3
Sep 2024,7.58,3.88,3.40,9.46,2.79,4.73,4.81,3.07,3.99,3.48,...,3.55,3.93,1.97,42.93,5.62,5.21,9.71,2024,Sep,3


In [31]:
# save as parquet file
new_month_df.to_parquet("../../data/month_mortgage_interest_values.parquet", index=False)